In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_validate


In [ ]:
train_df = pd.read_csv(r"c:\Users\HP\OneDrive\Masaüstü\yzta-2026-datathon\train.csv")
test_df = pd.read_csv(r"c:\Users\HP\OneDrive\Masaüstü\yzta-2026-datathon\test_x.csv")


In [ ]:
# 1. Veri Görselleştirme ve Keşifçi Veri Analizi (EDA)
# Yaş ve Bilişsel Performans İlişkisi (Keman Grafiği - Violin Plot)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")

plt.figure(figsize=(12, 6))

train_df['yas_grubu'] = pd.cut(train_df['yas'], bins=[18, 30, 45, 60], labels=['Genç (18-30)', 'Orta Yaş (31-45)', 'İleri Yaş (46-60)'])
sns.violinplot(x='yas_grubu', y='bilissel_performans_skoru', data=train_df, inner="quartile", palette="pastel")
plt.title('Yaş Gruplarına Göre Bilişsel Performans Dağılımı', fontsize=14, pad=15)
plt.xlabel('Yaş Grubu', fontsize=12)
plt.ylabel('Bilişsel Performans Skoru', fontsize=12)
plt.show()
train_df.drop('yas_grubu', axis=1, inplace=True) # Analiz sonrası geçici sütunu siliyoruz

In [ ]:
# Uyku Kalitesi ve Stres Skoru Etkileşimi (Hexbin Plot)
plt.figure(figsize=(10, 6))
plt.hexbin(train_df['uykuya_dalma_suresi_dk'], train_df['stres_skoru'], gridsize=30, cmap='Blues', mincnt=1)
cb = plt.colorbar(label='Gözlem Frekansı')
plt.title('Uykuya Dalma Süresi ve Stres Skoru Yoğunluk Analizi', fontsize=14, pad=15)
plt.xlabel('Uykuya Dalma Süresi (Dakika)', fontsize=12)
plt.ylabel('Stres Skoru', fontsize=12)
plt.show()

In [ ]:
# Mesleklere Göre Hedef Değişkenin Ortalaması (Bar Plot)
plt.figure(figsize=(14, 7))
meslek_siralamasi = train_df.groupby('meslek')['bilissel_performans_skoru'].mean().sort_values(ascending=False).index
sns.barplot(x='meslek', y='bilissel_performans_skoru', data=train_df, order=meslek_siralamasi, errorbar='sd', capsize=.1, palette='mako')
plt.title('Meslek Gruplarına Göre Ortalama Bilişsel Performans', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Meslek', fontsize=12)
plt.ylabel('Ortalama Bilişsel Performans Skoru', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Gecelik Uyanma Sayısının Hedefe Etkisi (Boxplot)
plt.figure(figsize=(10, 6))
sns.boxplot(x='gecelik_uyanma_sayisi', y='bilissel_performans_skoru', data=train_df, palette='Set2')
plt.title('Gecelik Uyanma Sayısının Performans Üzerindeki Marjinal Etkisi', fontsize=14, pad=15)
plt.xlabel('Gecelik Uyanma Sayısı', fontsize=12)
plt.ylabel('Bilişsel Performans Skoru', fontsize=12)
plt.show()

In [ ]:
# Stres Skoru ve Bilişsel Performans Arasındaki Doğrusal İlişki (Scatter Plot)
plt.figure(figsize=(10, 6))
sns.scatterplot(x='stres_skoru', y='bilissel_performans_skoru', data=train_df, alpha=0.5, color='coral')
plt.title('Stres Skoru ve Bilişsel Performans Arasındaki İlişki', fontsize=14, pad=15)
plt.xlabel('Stres Skoru', fontsize=12)
plt.ylabel('Bilişsel Performans Skoru', fontsize=12)
plt.show()

In [ ]:
# 2. Özellik Mühendisliği (Feature Engineering)
def create_features(df):
    df = df.copy()
    
    # Uyku Kalitesi Bileşenleri
    if 'rem_yuzdesi' in df.columns and 'derin_uyku_yuzdesi' in df.columns:
        df['kaliteli_uyku_yuzdesi'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
        df['hafif_uyku_yuzdesi'] = 100 - df['kaliteli_uyku_yuzdesi']
    
    # Negatif Etkenler Çarpanı
    if 'stres_skoru' in df.columns and 'uyku_oncesi_kafein_mg' in df.columns:
        df['stres_kafein_yuku'] = df['stres_skoru'] * (df['uyku_oncesi_kafein_mg'] + 1)
        
    if 'uyku_oncesi_ekran_suresi_dk' in df.columns and 'uykuya_dalma_suresi_dk' in df.columns:
        df['ekran_uyku_gecikmesi_etkisi'] = df['uyku_oncesi_ekran_suresi_dk'] * df['uykuya_dalma_suresi_dk']
        
    # Fiziksel Durum ve Aktivite
    if 'gunluk_adim_sayisi' in df.columns and 'vucut_kitle_indeksi' in df.columns:
        df['aktiflik_skoru'] = df['gunluk_adim_sayisi'] / (df['vucut_kitle_indeksi'] + 1)
        
    return df

print("Özellik mühendisliği uygulanıyor...")
train_df = create_features(train_df)
test_df = create_features(test_df)

In [ ]:
# Bağımsız (X) ve Bağımlı (y) Değişkenleri Ayırma
X = train_df.drop(['id', 'bilissel_performans_skoru'], axis=1)
y = train_df['bilissel_performans_skoru']

test_ids = test_df["id"]
X_test_comp = test_df.drop(['id'], axis=1)

In [ ]:
# 3. Veri Ön İşleme (Preprocessing) Pipeline Kurulumu
kategorik_sutunlar = X.select_dtypes(include=['object', 'category']).columns.tolist()
sayisal_sutunlar = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

sayisal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

kategorik_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Eksik')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', sayisal_transformer, sayisal_sutunlar),
        ('cat', kategorik_transformer, kategorik_sutunlar)
    ])

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

In [ ]:
catboost = Pipeline([
    ("prep", preprocessor),
    ("model", CatBoostRegressor(random_state=42))
])

lgbm = Pipeline([
    ("prep", preprocessor), 
    ("model", LGBMRegressor(random_state=42))
])

xgboost = Pipeline([
    ("prep", preprocessor), 
    ("model", XGBRegressor(random_state=42, eval_metric='rmse'))        
])

In [ ]:
models = {
    "catboost": catboost,
    "LightGBM": lgbm,
    "xgboost_pipeline": xgboost
}

In [ ]:
results = []

print("Modeller Eğitiliyor...")
for name, model in models.items():
    # bütün modeller için çapraz doğrulama
    cv_results = cross_validate(
        model, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False
    )
    
    rmse = -cv_results["test_RMSE"].mean()
    mae = -cv_results["test_MAE"].mean()
    r2 = cv_results["test_R2"].mean()
    
    results.append({
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })
    print(f"{name} tamamlandı.")


results_df = pd.DataFrame(results).sort_values("RMSE", ascending=True) # Hata ne kadar düşükse o kadar iyi
results_df

In [ ]:
# CATBOOST İÇİN DOĞRU OPTUNA OBJECTIVE FONKSİYONU
cat_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", CatBoostRegressor(random_state=42, verbose=0)) # verbose=0 ile eğitim sırasında gereksiz çıktıları kapatıyoruz
])

def objective_cat(trial):
    params = {
        # Ağaç sayısını artırıyoruz (Önceki en iyi 760'tı, aralığı genişlettik)
        "model__iterations": trial.suggest_int("iterations", 500, 2000),
        
        # En iyi LR 0.036 civarındaydı, odaklanmış bir aralık veriyoruz
        "model__learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        
        # CatBoost için derinliği 8 ile sınırlamak eğitim süresini ve genellenebilirliği korur
        "model__depth": trial.suggest_int("depth", 4, 8),
        
        # YENİ EKLENDİ: Aşırı öğrenmeyi engelleyen çok kritik L2 Regülarizasyonu
        "model__l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 25.0, log=True),
        
        # YENİ EKLENDİ: Ağaç yapısındaki varyansı düşürmek için rassallık gücü
        "model__random_strength": trial.suggest_float("random_strength", 0.1, 10.0, log=True),
        
        "model__rsm": trial.suggest_float("rsm", 0.5, 1.0),
        
        # Önceki model 48'de sınıra takılmıştı, üst sınırı 100'e çektik
        "model__min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 100),
        
        # Subsample parametresinin doğru çalışması için bootstrap tipi Bernoulli olmalı
        "model__bootstrap_type": "Bernoulli",
        "model__subsample": trial.suggest_float("subsample", 0.6, 1.0)
    }

    # Parametreleri pipeline'a uygula
    cat_pipeline.set_params(**params)

    # Cross validation skoru (Negatif RMSE dönüyor)
    score = cross_val_score(
        cat_pipeline,
        X,
        y,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

# Optuna study
study_cat = optuna.create_study(direction="maximize")

study_cat.optimize(objective_cat, n_trials=50)



print(f"En iyi parametreler: {study_cat.best_params}")
print(f"En iyi RMSE Skoru: {-study_cat.best_value:.4f}")

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMRegressor

# Pipeline
lgb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LGBMRegressor(random_state=42))
])

# Güncellenmiş Objective Fonksiyonu
def objective_lgb(trial):

    params = {
        # Sınır 1000'den 2500'e çıkarıldı çünkü model 814'ü seçmişti, daha fazlasına ihtiyaç duyabilir.
        "model__n_estimators": trial.suggest_int("n_estimators", 500, 2500),

        # En iyi learning rate 0.025 civarındaydı, aralığı bu bölgeye daralttık.
        "model__learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),

        # Model düşük derinlik (4) sevdiği için aralığı daraltıp sığ ağaçlara odaklandık.
        "model__max_depth": trial.suggest_int("max_depth", 3, 8),

        # Max depth 3-8 arası olduğu için 255 yaprak çok fazlaydı, gerçeğe daha uygun bir aralık.
        "model__num_leaves": trial.suggest_int("num_leaves", 15, 100),

        "model__subsample": trial.suggest_float("subsample", 0.6, 1.0),
        
        "model__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),

        "model__min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
        
        # YENİ: Aşırı öğrenmeyi engellemek için L1 ve L2 regülarizasyonu eklendi
        "model__reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "model__reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True)
    }

    lgb_pipeline.set_params(**params)

    score = cross_val_score(
        lgb_pipeline,
        X,
        y,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

# Optuna study
study_lgb = optuna.create_study(direction="maximize")

study_lgb.optimize(objective_lgb, n_trials=50)

# Sonuçlar
print("Best RMSE:", -study_lgb.best_value)
print("Best params:", study_lgb.best_params)

In [ ]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor

# Pipeline
xgb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", XGBRegressor(
        random_state=42,
        objective="reg:squarederror",
        verbosity=0
    ))
])

# Güncellenmiş Objective Fonksiyonu
def objective_xgb(trial):

    params = {
        # n_estimators sınırı 2500'e çıkarıldı (Önceki en iyi 925'ti).
        "model__n_estimators": trial.suggest_int("n_estimators", 500, 2500),
        
        "model__learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        
        # Önceki çalışmada max_depth 4 seçilmişti, sığ ağaçları zorlamak daha iyi sonuç verecek.
        "model__max_depth": trial.suggest_int("max_depth", 3, 8),
        
        "model__subsample": trial.suggest_float("subsample", 0.6, 1.0),
        
        "model__colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
        
        "model__min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        
        # Gamma için önceki en iyi değer ~4.0 olduğu için üst sınır genişletildi
        "model__gamma": trial.suggest_float("gamma", 1.0, 10.0),
        
        # YENİ: L1 ve L2 regülarizasyonu
        "model__reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "model__reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True)
    }

    xgb_pipeline.set_params(**params)

    score = cross_val_score(
        xgb_pipeline,
        X,
        y,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    ).mean()

    return score

# Optuna study
study_xgb = optuna.create_study(direction="maximize")

study_xgb.optimize(objective_xgb, n_trials=50)

print(f"En iyi parametreler: {study_xgb.best_params}")
print(f"En iyi RMSE Skoru: {-study_xgb.best_value:.4f}")

In [ ]:
print("Test verisi boyutu:", X_test_comp.shape)

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline

# 1. OPTUNA SONUÇLARINI ÇEKME
# study_lgb.best_params bize {'n_estimators': 800, 'learning_rate': 0.05, ...} gibi bir sözlük (dict) döner.
# Bu parametreleri doğrudan LGBMRegressor'ın içine ** kwargs mantığı ile açarak vereceğiz.

best_lgb_params = study_lgb.best_params
best_cat_params = study_cat.best_params
best_xgb_params = study_xgb.best_params

from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold

# 1. Base Modeller (Optuna parametreleri ile)
cat_pipeline = Pipeline([
    ("prep", preprocessor), 
    ("model", CatBoostRegressor(**best_cat_params, random_state=42, verbose=0))
])

lgb_pipeline = Pipeline([
    ("prep", preprocessor), 
    ("model", LGBMRegressor(**best_lgb_params, random_state=42)) 
])

xgb_pipeline = Pipeline([
    ("prep", preprocessor), 
    ("model", XGBRegressor(**best_xgb_params, random_state=42, verbosity=0))
])

estimators = [
    ('cat', cat_pipeline),
    ('lgb', lgb_pipeline),
    ('xgb', xgb_pipeline)
]

# 2. Stacking Kurulumu (Target Transformation kaldırıldı)
cv_stacking = KFold(n_splits=5, shuffle=True, random_state=42)
meta_model = RidgeCV(alphas=[0.1, 1.0, 10.0, 50.0, 100.0])

stacked_model = StackingRegressor(
    estimators=estimators,
    final_estimator=meta_model, 
    cv=cv_stacking, 
    n_jobs=-1
)

# 3. Model Eğitimi ve Tahmin
print("TargetEncoder'lı Stacking modeli eğitiliyor...")
stacked_model.fit(X, y)

print("Test verisi üzerinden tahminler üretiliyor...")
y_pred_comp = stacked_model.predict(X_test_comp)

In [ ]:
# 6. Sonuçları DataFrame'e dönüştürme
submission = pd.DataFrame({
    "id": test_ids,                             
    "bilissel_performans_skoru": y_pred_comp      
})

# 7. Dosyayı aynı klasöre CSV olarak kaydetme
kayit_yolu = r"c:\Users\HP\OneDrive\Masaüstü\yzta-2026-datathon\submission_team100.csv"
submission.to_csv(kayit_yolu, index=False)

print(f"Tahmin dosyası başarıyla oluşturuldu!\nDosya yolu: {kayit_yolu}")
print("\nOluşturulan dosyanın ilk 5 satırı:")
print(submission.head())